# Waxal ASR Colab Baseline Runner

Purpose: run a disciplined smoke test or zero-shot Whisper baseline on free Colab. Keep this notebook thin; source code lives in the repo.

## 1. Runtime Check

Use `Runtime > Change runtime type > GPU` before running model inference.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone Repo

Clone the published GitHub repository into the Colab runtime.

In [ ]:
REPO_URL = 'https://github.com/Maziko-M98/waxal-asr-challenge.git'
PROJECT_DIR = '/content/waxal_asr_challenge'

!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

## 3. Install Dependencies

This installs the project and ASR dependencies into the Colab runtime.

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -e . -q
!pip install -r requirements.txt -q

## 4. Provide Zindi CSV Files

Upload the three Zindi CSV files when prompted. They are intentionally not committed to Git.

In [ ]:
from google.colab import files
from pathlib import Path

data_dir = Path('data/zindi')
data_dir.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, payload in uploaded.items():
    target = data_dir / name
    target.write_bytes(payload)
    print('wrote', target, len(payload), 'bytes')

## 5. EDA And Dry Run

In [ ]:
!python -m waxal_asr.inspect_zindi_csv --data-dir data/zindi
!python -m waxal_asr.eda_report --data-dir data/zindi --out-dir reports/eda
!python -m waxal_asr.infer_whisper --config configs/baselines/zero_shot_whisper_large_v3_turbo.json --dry-run

## 6. Smoke Inference

Run only 20 examples first. Inspect the output before full inference.

In [ ]:
!python -m waxal_asr.infer_whisper \
  --config configs/baselines/zero_shot_whisper_large_v3_turbo.json \
  --max-samples 20 \
  --output submissions/smoke_20_zero_shot_whisper.csv \
  --raw-predictions submissions/raw/smoke_20_zero_shot_whisper_raw.csv

!python - <<'PY'
import pandas as pd
df = pd.read_csv('submissions/smoke_20_zero_shot_whisper.csv')
print(df.shape)
print(df.head())
PY

## 7. Full Baseline

Run this only after the smoke test succeeds and there is enough runtime left.

In [ ]:
# !python -m waxal_asr.infer_whisper \
#   --config configs/baselines/zero_shot_whisper_large_v3_turbo.json \
#   --raw-predictions submissions/raw/baseline_001_zero_shot_whisper_large_v3_turbo_raw.csv